# Projecte: BICITIERRA - Neteja de dades de sales history

In [ ]:
# En primer lloc anem a carregar les llibreries necessàries i llegir el dataset de vendes.
import pandas as pd

# tin en compte que la ruta el dataset por variar en funció de com tingues organitzada la carpeta del projecte
df = pd.read_csv('./dades/sales_history.csv')
#3. Després mostre les 15 primeres línies.
df.head(15)

### Pas 1: Descarregar les dades i construir el DataFrame

In [4]:
# Mostrem l'estructura del DataFrame (noms de les columnes i tipus de dades)
print("Shape:", df.shape) # Mostra la quantitat de files i columnes
print("\nColumns and data types:") # Mostra els noms de les columnes i els tipus de dades
print(df.dtypes)


Shape: (1188, 17)

Columns and data types:
month                      str
channel                    str
product_line               str
region                     str
marketing_spend        float64
avg_discount           float64
stock_units              int64
website_sessions       float64
sales_units              int64
revenue                float64
marketing_budget       float64
planned_discount       float64
list_price             float64
campaign                   str
days_in_month            int64
closing_stock_units      int64
stockout                 int64
dtype: object


### Pas 2: Preprocessament de les dades

Abans de començar a entrenar els models, és important fer un preprocessament de les dades. 

- Comproveu si hi ha valors nuls i, si n'hi ha, decidiu com tractar-los (per exemple, eliminant les files amb valors nuls o omplint-los amb la mitjana o la mediana)
- Comproveu si hi ha variables categòriques (no numèriques). Decidiu què fer amb elles.
- Mireu si hi ha molta diferència entre les magnituds de les diferents característiques i decidiu si seria convenient normalitzar o estandarditzar algunes d'elles.

In [5]:
# Comprovem si hi ha valors nuls al DataFrame
print("\nNull values in each column:")
print(df.isnull().sum())


Null values in each column:
month                   0
channel                 0
product_line            0
region                  0
marketing_spend        18
avg_discount           24
stock_units             0
website_sessions       18
sales_units             0
revenue                 0
marketing_budget       12
planned_discount       12
list_price              0
campaign                0
days_in_month           0
closing_stock_units     0
stockout                0
dtype: int64


In [ ]:
# Tractem els nulls
cols = ["marketing_spend", "avg_discount", "website_sessions",
        "marketing_budget", "planned_discount"]

# Mira abans la distribució
print(df[cols].describe())

# En average_discount i planned_discount hi ha valors que són 0.00, per tant anem a asumir que en estes columnes el nul equival a 0.
# Pel que fa a la resta de columnes considerem que sí podriem sustituir els nuls per la mediana.

# La mitjana és la suma de tots els valors dividida pel nombre de valors. 
# La mediana és el valor que queda al mig quan ordenes les dades de menor a major.

       marketing_spend  avg_discount  website_sessions  marketing_budget  \
count      1170.000000   1164.000000       1170.000000       1176.000000   
mean       2851.213812      0.076083       5466.698291       2863.435748   
std        1084.577937      0.045830       1568.814205       1072.068280   
min         920.680000      0.000000       2120.000000       1014.670000   
25%        1999.882500      0.045000       4158.000000       1985.257500   
50%        2831.765000      0.063000       5506.500000       2833.150000   
75%        3601.235000      0.097250       6655.750000       3648.410000   
max        6059.990000      0.219000      10261.000000       5664.070000   

       planned_discount  
count       1176.000000  
mean           0.076283  
std            0.044783  
min            0.000000  
25%            0.045000  
50%            0.063000  
75%            0.096250  
max            0.206000  


In [ ]:
# Descomptes: nul = sense descompte
df[["avg_discount", "planned_discount"]] = df[["avg_discount", "planned_discount"]].fillna(0)

# Resta: mediana
for c in ["marketing_spend", "website_sessions", "marketing_budget"]:
    df[c] = df[c].fillna(df[c].median())

# Comprovació
print(df.isnull().sum())

In [ ]:
# Mostrem dades estadístiques de cada columna. Això ens servirà també per veure quins són els valors possibles
print(df.describe())

                     month  marketing_spend  avg_discount  stock_units  \
count                 1188      1188.000000   1188.000000  1188.000000   
mean   2023-09-16 04:00:00      2850.919133      0.074546   125.931818   
min    2021-01-01 00:00:00       920.680000      0.000000    17.000000   
25%    2022-05-01 00:00:00      2009.542500      0.044000    80.000000   
50%    2023-09-16 00:00:00      2831.765000      0.063000   113.000000   
75%    2025-02-01 00:00:00      3593.985000      0.096000   161.000000   
max    2026-06-01 00:00:00      6059.990000      0.219000   356.000000   
std                    NaN      1076.325725      0.046611    60.500924   

       website_sessions  sales_units        revenue  marketing_budget  \
count       1188.000000  1188.000000    1188.000000       1188.000000   
mean        5467.301347   100.244108   70409.271120       2863.129832   
min         2120.000000    17.000000   18446.230000       1014.670000   
25%         4196.000000    63.000000   50

## Tractament de la columna month

In [ ]:
# Anem a crear dos columnes més: Year i month_num
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")

df["year"] = df["month"].dt.year
df["month_num"] = df["month"].dt.month   # 1-12, per a l'estacionalitat

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1188 entries, 0 to 1187
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   month                1188 non-null   datetime64[us]
 1   channel              1188 non-null   str           
 2   product_line         1188 non-null   str           
 3   region               1188 non-null   str           
 4   marketing_spend      1188 non-null   float64       
 5   avg_discount         1188 non-null   float64       
 6   stock_units          1188 non-null   int64         
 7   website_sessions     1188 non-null   float64       
 8   sales_units          1188 non-null   int64         
 9   revenue              1188 non-null   float64       
 10  marketing_budget     1188 non-null   float64       
 11  planned_discount     1188 non-null   float64       
 12  list_price           1188 non-null   float64       
 13  campaign             1188 non-null   str    

In [ ]:
cat_cols = ["channel", "product_line", "region", "campaign"]
# Hi ha quatre variables categòriques: channel, product_line, region i campaign.
# Comprovem quins valors diferents hi ha per a cada columna, per si s'ha de transformar a numèrica

for c in cat_cols:
    print(f"\nUnique values in '{c}' column:")
    print(df[c].unique())


# També volem saber quants hi ha al dataset

for c in cat_cols:
    print(f"\nValue counts for '{c}' column:")
    print(df[c].value_counts())


Unique values in 'channel' column:
<StringArray>
['store', 'ecommerce']
Length: 2, dtype: str

Unique values in 'product_line' column:
<StringArray>
['urban', 'mountain', 'electric']
Length: 3, dtype: str

Unique values in 'region' column:
<StringArray>
['coast', 'inland', 'metro']
Length: 3, dtype: str

Unique values in 'campaign' column:
<StringArray>
['none', 'spring', 'summer', 'black_friday', 'christmas']
Length: 5, dtype: str

Value counts for 'channel' column:
channel
store        594
ecommerce    594
Name: count, dtype: int64

Value counts for 'product_line' column:
product_line
urban       396
mountain    396
electric    396
Name: count, dtype: int64

Value counts for 'region' column:
region
coast     396
inland    396
metro     396
Name: count, dtype: int64

Value counts for 'campaign' column:
campaign
none            906
spring           80
christmas        70
summer           69
black_friday     63
Name: count, dtype: int64


In [21]:
# Quantes files completament duplicades hi ha
print(df.duplicated().sum())

0


In [25]:

# Veure-les (keep=False mostra totes les còpies, no només les repetides)
print(df[df.duplicated(keep=False)])

Empty DataFrame
Columns: [month, channel, product_line, region, marketing_spend, avg_discount, stock_units, website_sessions, sales_units, revenue, marketing_budget, planned_discount, list_price, campaign, days_in_month, closing_stock_units, stockout, year, month_num]
Index: []


In [26]:
key = ["month", "channel", "product_line", "region"]
print(df.duplicated(subset=key).sum())

0


In [22]:
# Anem a transformar les columnes categòriques a numèriques amb la tècnica one-hot.
# La funció get_dummies de pandas crea una nova columna per a cada categoria de l'atribut categòric
# amb valors binaris (1 si la fila pertany a eixa categoria, 0 si no pertany)
df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True)  # drop_first=True lleva una columna per variable, que és el que convé a models lineals perquè la que falta ja queda implícita.

In [19]:
# Mostrem dades estadístiques de cada columna. Això ens servirà també per veure quins són els valors possibles
print(df.describe())

                     month  marketing_spend  avg_discount  stock_units  \
count                 1188      1188.000000   1188.000000  1188.000000   
mean   2023-09-16 04:00:00      2850.919133      0.074546   125.931818   
min    2021-01-01 00:00:00       920.680000      0.000000    17.000000   
25%    2022-05-01 00:00:00      2009.542500      0.044000    80.000000   
50%    2023-09-16 00:00:00      2831.765000      0.063000   113.000000   
75%    2025-02-01 00:00:00      3593.985000      0.096000   161.000000   
max    2026-06-01 00:00:00      6059.990000      0.219000   356.000000   
std                    NaN      1076.325725      0.046611    60.500924   

       website_sessions  sales_units        revenue  marketing_budget  \
count       1188.000000  1188.000000    1188.000000       1188.000000   
mean        5467.301347   100.244108   70409.271120       2863.129832   
min         2120.000000    17.000000   18446.230000       1014.670000   
25%         4196.000000    63.000000   50

## Comprovem outliners

In [27]:
# Columnes numèriques, sense les de calendari ni la binària
columnes_numeriques = df.select_dtypes(include="number").columns.drop(
    ["year", "month_num", "days_in_month", "stockout"]
)

print("Nombre d'outliers per columna (criteri IQR):\n")

for col in columnes_numeriques:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lim_inferior = Q1 - 1.5 * IQR
    lim_superior = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lim_inferior) | (df[col] > lim_superior)).sum()

    print(f"{col}: {n_outliers}")

Nombre d'outliers per columna (criteri IQR):

marketing_spend: 3
avg_discount: 42
stock_units: 12
website_sessions: 0
sales_units: 8
revenue: 36
marketing_budget: 0
planned_discount: 37
list_price: 0
closing_stock_units: 37


In [28]:
# Límit superior de l'IQR per a avg_discount
Q1, Q3 = df["avg_discount"].quantile([0.25, 0.75])   # quartils 25 % i 75 %
lim_sup = Q3 + 1.5 * (Q3 - Q1)                       # tot el que supere això és outlier

# Ens quedem només amb les files que el superen
outl = df[df["avg_discount"] > lim_sup]

# Comptem a quina campanya pertanyen eixes files
print(outl["campaign"].value_counts())

campaign
black_friday    18
spring          10
christmas        9
summer           5
Name: count, dtype: int64


In [ ]:
# Els 42 outliers de avg_discount són de campanya. Suma 18 + 10 + 9 + 5 = 42, i cap és de none.

# Vol dir que no són errors, sinó descomptes reals de promocions. Black Friday en té més (18), que és el més lògic. Com que hi ha només 63 files de Black Friday, un terç d’elles (18) té un descompte alt, però dins de la seua campanya és normal.

# Què fer: no llevar-los. Ja tens la informació en campaign, que després passaràs a one-hot, i així el model podrà relacionar “campanya + descompte alt” amb les vendes.

# Pots repetir el mateix amb planned_discount, que segurament done un resultat paregut, i amb revenue agrupant per product_line en lloc de campaign.
